# LLM 文本增强 — DeepSeek API

    仅根据 **title + domain** 生成简短增强属性，不传 description
    输出 ~50-80 tokens，适合拼入 CLIP (77 tok) 上下文
    description 原样保留，Long-CLIP 编码时拼入

In [ ]:
import os
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import threading

DEEPSEEK_API_KEY = ""
DEEPSEEK_BASE_URL = "https://api.deepseek.com"
MODEL = "deepseek-v4-pro"       # V4 Pro

META_CSV = "final/item_meta_merged.csv"
OUTPUT_CSV = "final/item_meta_llm_enhanced.csv"

BATCH_SAVE = 500
TEMPERATURE = 0.1
MAX_TOKENS = 80
MAX_WORDS = 50
MAX_CHARS  = 500
NUM_THREADS = 20
SLEEP_RATE_LIMIT = 5
TEST_MODE = False

write_lock = threading.Lock()
client = OpenAI(api_key=DEEPSEEK_API_KEY, base_url=DEEPSEEK_BASE_URL)

### Prompt 设计


In [10]:
def build_prompt(domain, title):
    domain_name = "Movie/TV" if domain == 0 else "Book"
    return f"""You are helping build a cross-domain recommendation system that matches {domain_name}s to users across different media types.

Given the {domain_name} title "{title}", generate a concise semantic profile that captures:

- Core genre and subgenre (be specific, not just "drama" but "psychological crime drama")
- Key themes and narrative style (e.g. "coming-of-age story with nonlinear storytelling")
- Intended audience and the emotional experience it delivers (e.g. "for readers who enjoy slow-burn atmospheric horror")

Write 2-3 natural sentences. Avoid vague terms. Make every word count for distinguishing this item from others in the same genre. Keep under {MAX_WORDS} words total.

Output only the profile text, nothing else."""

In [11]:
# ==================== API 调用 ====================

def call_deepseek(prompt):
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": "You are a professional cross-domain item analyst."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
                extra_body={"thinking": {"type": "disabled"}}   # 关掉思维链，省钱
            )
            text = response.choices[0].message.content.strip()
            return text[:MAX_CHARS]
        except Exception as e:
            err = str(e)
            if "rate_limit" in err.lower() or "429" in err:
                print(f"  限流，等待 {SLEEP_RATE_LIMIT}s...")
                time.sleep(SLEEP_RATE_LIMIT)
            elif attempt < 2:
                time.sleep(2)
            else:
                print(f"  API 失败(尝试{attempt+1}/3): {err[:100]}")
    return ""

In [12]:
# ==================== 主流程（多线程） ====================

print("=" * 60)
print(f"LLM 文本增强（{NUM_THREADS} 线程并发）")
print("=" * 60)

meta_df = pd.read_csv(META_CSV)
print(f"总物品数: {len(meta_df)}")

if TEST_MODE:
    meta_df = meta_df.head(10)
    print("测试模式：仅处理前 10 条")
    NUM_THREADS = 2

# 断点续传
if os.path.exists(OUTPUT_CSV):
    saved = pd.read_csv(OUTPUT_CSV)
    done_mask = saved["llm_enhanced_text"].notna() & (saved["llm_enhanced_text"] != "")
    if done_mask.any():
        meta_df = pd.merge(
            meta_df,
            saved[["parent_asin", "llm_enhanced_text"]].loc[done_mask],
            on="parent_asin", how="left"
        )
        print(f"断点续传：已有 {done_mask.sum()} 条完成")
    else:
        meta_df["llm_enhanced_text"] = ""
else:
    meta_df["llm_enhanced_text"] = ""

to_process = meta_df[
    meta_df["llm_enhanced_text"].isna() | (meta_df["llm_enhanced_text"] == "")
]
print(f"待处理: {len(to_process)}")

if len(to_process) == 0:
    print("全部已完成！")
else:
    # 构造任务列表
    tasks = [(idx, row["domain"], row["title"]) for idx, row in to_process.iterrows()]
    completed = 0

    def process_one(task):
        idx, domain, title = task
        prompt = build_prompt(domain, title)
        enhanced = call_deepseek(prompt)
        return idx, enhanced

    with ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
        futures = {executor.submit(process_one, t): t for t in tasks}
        for future in tqdm(as_completed(futures), total=len(futures), desc="生成"):
            idx, enhanced = future.result()
            meta_df.at[idx, "llm_enhanced_text"] = enhanced
            completed += 1

            if completed % BATCH_SAVE == 0:
                with write_lock:
                    meta_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

    meta_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    print(f"完成 -> {OUTPUT_CSV}")

LLM 文本增强（20 线程并发）
总物品数: 43528
待处理: 43528


生成: 100%|██████████| 43528/43528 [1:42:48<00:00,  7.06it/s]  


完成 -> final/item_meta_llm_enhanced.csv


In [13]:
# ==================== 结果预览 ====================
df = pd.read_csv(OUTPUT_CSV)
has_enhanced = df["llm_enhanced_text"].notna() & (df["llm_enhanced_text"] != "")
print(f"增强完成: {has_enhanced.sum()}/{len(df)}")

print("\n--- 样例 ---")
sample = df[has_enhanced].sample(min(5, has_enhanced.sum()), random_state=42)
for _, r in sample.iterrows():
    domain = "movie" if r["domain"] == 0 else "book"
    print(f"\n[{domain}] {r['title'][:60]}")
    print(f"  增强 ({len(r['llm_enhanced_text'])} chars): {r['llm_enhanced_text'][:200]}")

df['enhanced_len'] = df['llm_enhanced_text'].fillna('').astype(str).str.len()
print(f"\n增强文本长度: min={df['enhanced_len'].min():.0f} max={df['enhanced_len'].max():.0f} "
      f"avg={df['enhanced_len'].mean():.0f} median={df['enhanced_len'].median():.0f}")
print(f"> 500 chars 的条数: {(df['enhanced_len'] > 500).sum()}")

增强完成: 43528/43528

--- 样例 ---

[movie] Tyler Perry's Boo! A Madea Halloween [DVD]
  增强 (281 chars): A comedic horror film blending slapstick humor with supernatural pranks. It explores family chaos and generational clashes through a farcical Halloween narrative. Designed for audiences seeking lighth

[movie] George A. Romero's Land of the Dead
  增强 (225 chars): Post-apocalyptic zombie survival horror with social allegory. Explores class warfare and human tribalism amid undead siege. Delivers visceral action and bleak tension for fans of politically charged, 

[book] The Regulators
  增强 (278 chars): A suburban horror thriller blending supernatural chaos with visceral violence. It explores reality distortion and childhood innocence corrupted, using parallel-universe tension and rapid-fire pacing. 

[book] Vienna Secrets: A Max Liebermann Mystery
  增强 (294 chars): A historical psychological crime thriller set in early 1900s Vienna. It blends forensic investigation with psychoanalytic them

## 长度控制（三层保障）

| 层级 | 机制 | 作用 |
|:--:|------|------|
| ① | `max_tokens=80` | API 硬限制，模型无法超出 |
| ② | prompt: `Keep under 50 words` | 引导模型精简 |
| ③ | `text[:500]` 截断 | 代码兜底，万无一失 |

## 后续编码

```python
# CLIP (77 tok)
text_clip = df['title'] + ' | ' + df['llm_enhanced_text']

# Long-CLIP (248 tok)
text_longclip = df['title'] + ' | ' + df['llm_enhanced_text'] + ' | ' + df['description'].fillna('')[:600]
```